# Hypothesis Testing with Monte Carlo Simulation

## Learning aims

- Use simulation to reason about whether observed data is surprising
- Define and implement the four steps of null hypothesis testing
- Compute and interpret a p-value
- Understand the core concepts of hypothesis testing precisely

## Motivation: back to the suspicious coin

In the last lecture we looked at `coin2()` — a coin that turned out to give heads 80% of the time. We could see this because we had access to the source code.

But what if you cannot look inside? Imagine a friend flips a coin 10 times and gets **8 heads**. You start to wonder: is this a fair coin?

This is the core question of **hypothesis testing**: given some observed data, how surprising would it be if the data came from a process we believe to be "normal"?


## The four steps

We will answer this by following four steps. The starting point is always the **null hypothesis** (written H₀): a precise statement of what "nothing unusual is happening" means for your specific problem. Everything else follows from it.

1. **Define a null model** — what does "normal" look like?
2. **Compute a test statistic** — summarise the observed data in one number
3. **Generate a null distribution** — simulate what that number looks like under the null model
4. **Compute a p-value** — how often does the null model produce a result like ours?

# Step 1: Define a null model

The null model encodes our default assumption — the process we assume is operating unless we find evidence against it. It is the computational expression of the null hypothesis.

For this example, the null hypothesis is:

> **H₀: The coin is fair — each flip produces heads with probability 0.5, independently of all other flips.**

In code this means generating flips with `p_head=0.5`.

In code this is `weighted_coin` and `toss_coin_n_times` from the previous lecture, with `p_head=0.5`:


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def weighted_coin(p_head: float = 0.5) -> str:
    return 'H' if random.random() < p_head else 'T'

def toss_coin_n_times(p_head: float, n: int) -> list[str]:
    return [weighted_coin(p_head) for _ in range(n)]

# Under the null model:
toss_coin_n_times(p_head=0.5, n=10)

## Step 2: Compute a test statistic

We need a way to summarise our observed data as a single number. This is called the **test statistic**.

Since we are asking whether there are too many heads, the natural choice is simply: **the number of heads**.

In [ ]:
def test_statistic(flips: list[str]) -> int:
    return flips.count('H')

# Our observed data
observed_flips = ['H', 'H', 'T', 'H', 'H', 'H', 'T', 'H', 'H', 'T']
observed_stat = test_statistic(observed_flips)

print(f"Observed test statistic: {observed_stat} heads out of {len(observed_flips)} flips")

## Step 3: Generate a null distribution

This is the Monte Carlo step. We simulate the experiment many times under the null model and record the test statistic each time. The result is the **null distribution** — a picture of what our test statistic typically looks like when the coin really is fair.

This loop is the same pattern from the previous lecture, just with a new purpose:


In [ ]:
n_flips = 10
n_simulations = 10_000

null_distribution = [
    test_statistic(toss_coin_n_times(p_head=0.5, n=n_flips))
    for _ in range(n_simulations)
]

sns.histplot(null_distribution, discrete=True)
plt.axvline(observed_stat, color='red', linestyle='--', label=f'Observed: {observed_stat}')
plt.xlabel('Number of heads')
plt.ylabel('Count')
plt.title('Null distribution (fair coin, 10 flips)')
plt.legend()
plt.show()

The red line shows where our observation falls. Already we can see visually that 8 heads sits well into the tail.


## Step 4: Compute a p-value

Now we want a number that captures how surprising our observation is. We can read this directly from the null distribution: in what fraction of our simulated experiments did a fair coin produce as many heads as we observed?

But notice: if 8 heads makes us suspicious, then 9 or 10 heads would make us *more* suspicious — not less. So it makes sense to count all of those together. We ask: how often did the null model produce **8 or more** heads?

In [ ]:
p_value = sum(x >= observed_stat for x in null_distribution) / n_simulations
print(f"p-value: {p_value:.3f}")

This is the **p-value**: roughly 5% of fair-coin experiments produce 8 or more heads. In other words, this result would happen only 1 in 20 times by chance alone.


## Interpretation

A p-value of ~0.05 means: if the coin were truly fair, we would see 8 or more heads in about 5% of experiments. That is unusual enough to raise an eyebrow — but it does not *prove* the coin is biased.

By convention, a p-value below **0.05** is often described as *statistically significant*, meaning the result is surprising enough under the null model that we might consider rejecting it. But this threshold is a rule of thumb, not a law.

A p-value is **not**:
- The probability that the null model is true
- The probability that the result is due to chance
- A measure of how large or important the effect is

It is only the answer to: *how surprising is this data, assuming the null model?*

## Exercise: do antibiotics help after a bacterial throat infection?

You have data from a group of patients diagnosed with a bacterial throat infection. Some were given antibiotics, others were not. For each patient you have two pieces of information: whether they received antibiotics, and how many days it took them to recover.

In [ ]:
import numpy as np

received_antibiotics = np.array([True, False, True, True, False, True, False, False, True, False])
days_to_recovery = np.array([5, 9, 4, 6, 11, 5, 10, 8, 4, 12])

The question is: do patients who received antibiotics recover faster?

The null hypothesis is:

> **H₀: Antibiotics have no effect on recovery time — the label (antibiotic or not) is unrelated to how many days a patient takes to recover.**

In [ ]:
def compute_p_value(labels: np.ndarray, values: np.ndarray) -> float:
    pass

compute_p_value(received_antibiotics, days_to_recovery)

### Solution

A straightforward implementation puts everything in one function:

In [ ]:
def estimate_p_value(labels: np.ndarray, values: np.ndarray, n_simulations: int = 10_000) -> float:
    observed_stat = np.mean(values[labels]) - np.mean(values[~labels])

    null_distribution = []
    for _ in range(n_simulations):
        shuffled_labels = np.random.permutation(labels)
        stat = np.mean(values[shuffled_labels]) - np.mean(values[~shuffled_labels])
        null_distribution.append(stat)

    return np.mean(np.array(null_distribution) <= observed_stat)

estimate_p_value(received_antibiotics, days_to_recovery)

This works, but — as we saw in a previous lecture — it mixes together things that are conceptually different. Just as we separated the Monopoly simulation into a DGP and a generic estimator, we can do the same here. In fact, we can go further: the estimation side itself contains three distinct conceptual steps.

There are four parts in total:

1. **Null model** — the domain-specific DGP: what does one sample from a world where antibiotics have no effect look like?
2. **Test statistic** — how do we summarise the data from one sample into a single number?
3. **Null distribution** — run the null model many times and collect the test statistics
4. **p-value** — where does the observed statistic fall in that distribution?

If antibiotics have no effect, the label assigned to each patient is arbitrary. Shuffling the labels gives us one outcome — a complete dataset as it could have appeared under the null. The values (days to recovery) stay the same; only the group assignments change:


In [ ]:
def sample_from_null_model(labels: np.ndarray, values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return np.random.permutation(labels), values

**Test statistic**

The test statistic takes an outcome (a full dataset) and extracts a scalar — the difference in mean recovery time between the two groups:

In [ ]:
def compute_test_statistic(outcome: tuple) -> float:
    labels, values = outcome
    test_statistic = np.mean(values[labels]) - np.mean(values[~labels])
    return test_statistic

**Null distribution**

Run the null model many times, apply the test statistic to each outcome, and collect the results:

In [ ]:
def compute_null_distribution(labels: np.ndarray, values: np.ndarray, n_simulations: int = 10_000) -> np.ndarray:
    null_test_statistics = []
    for _ in range(n_simulations):
        outcome = sample_from_null_model(labels, values)
        test_statistic = compute_test_statistic(outcome)
        null_test_statistics.append(test_statistic)
    return np.array(null_test_statistics)

**p-value**

Finally, compare the observed statistic against the null distribution:

In [ ]:
def estimate_p_value(labels: np.ndarray, values: np.ndarray, n_simulations: int = 10_000) -> float:
    observed_stat = compute_test_statistic((labels, values))
    null_distribution = compute_null_distribution(labels, values, n_simulations)
    return np.mean(null_distribution <= observed_stat)

estimate_p_value(received_antibiotics, days_to_recovery)

### Note: the outcome here is a full dataset

In the coin example, one outcome from the null model was a simple scalar — the number of heads in 10 flips. In the Monopoly example from a previous lecture, one outcome was the number of throws to complete a lap. In both cases, the outcome was a single number that could be compared directly.

In the antibiotics example something different happened. One outcome from the null model was a **complete dataset** — a full pair of arrays `(labels, values)` representing all 10 patients with their (shuffled) group assignments and recovery times. This is not a simple value; it is the same shape and structure as the original observed data, just with the labels permuted.

This is in fact the general idea behind hypothesis testing: the null model does not produce a simplified summary — it produces **another version of your dataset as it could have been**, had the null hypothesis been true. In our case, H₀ says the labels are irrelevant, so any permutation of the labels is an equally valid dataset under H₀.

This is why we need the test statistic. The null model gives us a dataset, but we cannot directly ask "is this dataset more extreme than our observed dataset?" — that question has no clear answer when the outcome is a complex object. The test statistic compresses each dataset down to a single number — a specific characteristic we care about — so that comparison becomes possible.

The roles of each component can therefore be described precisely:

- The **null model** is a data generating process for *datasets* — each call produces one complete version of how the data could have looked under H₀
- The **test statistic** transforms each such dataset into a single scalar, extracting the one characteristic we want to compare
- The **null distribution** is the resulting distribution of that scalar under the null model — effectively a DGP for the test statistic itself
- The **p-value** tells us where the observed test statistic falls within that distribution

This structure makes hypothesis testing extremely general. The null distribution and p-value estimation are completely generic — the same code works for any problem. What changes between problems is only two things:

1. **The null hypothesis** — a precise statement of what "no effect" means in your domain
2. **The null model** — how to generate one dataset as it could have been under H₀ (domain-specific)
3. **The test statistic** — which characteristic of the dataset to extract and compare (problem-specific)

Everything else — `compute_null_distribution`, which runs the null model many times and collects test statistics, and `estimate_p_value`, which computes the fraction that are at least as extreme — is the same regardless of the problem. Once you have defined those two problem-specific pieces, the machinery does the rest.


## Exercise: how many flips do you need?

With only 10 flips, even a biased coin will sometimes look fair. Experiment with different numbers of flips and observe how the p-value changes when the true coin has `p_head=0.7`.


In [ ]:
# TODO: for each n in [10, 20, 50, 100]:
#   simulate observed_flips from weighted_coin(p_head=0.7)
#   compute the Monte Carlo p-value against the null model p=0.5
#   print n and the p-value

What do you notice? How does the number of observations affect our ability to detect that the coin is biased?


## Core concepts: precise definitions

Now that we have worked through an example, let us go back and define the concepts more carefully and generally.

### Null model (null hypothesis, H₀)

The null model describes the assumed default process — the world as it would be if nothing unusual were happening. In our case: a fair coin with P(head) = 0.5.

The null model is always something we can simulate from. We are not claiming it is true; we are asking how well the data is consistent with it.

### Test statistic

A test statistic is a function of the data that produces a single scalar value. It summarises the data in the dimension that is relevant to the question.

It must be scalar because we need to be able to order results and ask whether one is more extreme than another. A full list of flip outcomes (`['H', 'T', 'H', ...]`) cannot be ordered in a meaningful way. A count (8, 5, 3) can.

The choice of test statistic determines what the test is sensitive to. A bad choice can make a test blind to the very effect it is meant to detect.

### Null distribution

The null distribution is the probability distribution of the test statistic under the null model. We approximated it via Monte Carlo simulation — running many experiments under H₀ and collecting the resulting test statistic values.

The null distribution is the reference. It tells us what is normal. Without it we have no basis for calling any observation surprising.

### p-value

The p-value is the probability, under the null model, of obtaining a test statistic at least as extreme as the one we observed.

Equivalently: the fraction of values in the null distribution that are at least as extreme as the observed value.

In code:

In [ ]:
p_value = sum(x >= observed_stat for x in null_distribution) / n_simulations

Note that "at least as extreme" is phrased in terms of test statistic values, not simulations. In a one-sided test (as above) that means ≥ the observed value. In a two-sided test it means at least as far from the centre in either direction.


### Why "at least as extreme"?

This is not arbitrary. Consider the alternative: asking how often the null model produces *exactly* our observed value. For continuous data this probability is always zero, which is useless. Even for discrete data like coin flips, it gives the wrong answer: the probability of exactly 5 heads is also small, yet 5 heads is the most expected outcome.

What we actually want to measure is: *how far into the tail is our observation?* This requires aggregating all outcomes that are at least as surprising as ours. That is what "at least as extreme" does.

It also captures a natural intuition: if 8 heads is evidence against a fair coin, then 9 and 10 heads are *stronger* evidence, not weaker. A p-value should reflect that.


### Connection to the DGP and estimation framework

In a previous lecture we introduced the idea of separating a simulation into two parts: a **data generating process** that produces one sample from the mechanism of interest, and a **generic estimator** that runs the DGP many times and aggregates the results into a probability distribution.

Hypothesis testing has exactly the same two-part structure — but the estimation side is more involved.

The **DGP part** is the null model. In our example this is just `weighted_coin` — a function that produces one coin flip. It is domain-specific: it encodes the assumption that the coin is fair.

The **estimation part** is everything else: the test statistic, the null distribution, and the p-value. Together these form a structured procedure that takes raw samples from the DGP and converts them into a single summary of how surprising the observed data is. The extra structure compared to simple probability estimation is the test statistic step: because the DGP produces raw data (a list of flips) rather than a directly comparable outcome, we need to extract a characteristic from it first before we can build a distribution and compare.

So while the Monopoly estimator from the previous lecture produced a probability distribution over outcomes, the hypothesis testing estimator produces a single number — the p-value — that answers a more specific question about where the observed data falls within that distribution.


### A brief historical note

The p-value was formalised by **Ronald Fisher** in his 1925 book *Statistical Methods for Research Workers*. His canonical example was an experiment where a woman claimed she could tell by taste whether milk or tea had been poured into a cup first — structurally identical to our coin problem: define what random guessing looks like, simulate (or compute) the null distribution, see how often it matches or exceeds the observation.

Fisher proposed 0.05 as a convenient rule of thumb for "surprising enough to investigate further". He was explicit that it was not a universal law.

**Jerzy Neyman** and **Egon Pearson** later developed a complementary framework based on explicit alternative hypotheses, error rates, and decision rules (Type I and Type II errors). Modern statistics draws on both. We use Fisher's p-value framework here, but it is worth knowing that "hypothesis testing" is a family of related ideas developed over several decades and still actively debated.


## Analytical solutions and standard tests

Throughout this lecture we have estimated p-values by simulation: generate many outcomes from the null model, compute a test statistic for each, and count how many are at least as extreme as observed. This is the Monte Carlo approach.

In the previous lecture on probability distributions we saw that the same situation arises there too. We estimated probability distributions by simulation — but for many standard processes it turned out we could instead derive an exact mathematical formula. The binomial distribution gave us exact probabilities for coin tosses without any simulation at all.

The same is true for p-values. For many standard testing problems, mathematicians have derived exact analytical formulas that give the p-value directly, without any need to run simulations. These are typically faster to compute and give exact rather than approximate results.


### The coin example: exact p-value from the binomial distribution

Our coin test statistic was the number of heads in 10 flips. Under H₀ (fair coin), this follows a binomial distribution — which we already know from the previous lecture. The p-value is simply the probability of getting 8 or more heads under that distribution, which we can compute directly:

In [ ]:
from scipy.stats import binom

# P(X >= 8) = 1 - P(X <= 7) for X ~ Binomial(n=10, p=0.5)
exact_p = 1 - binom.cdf(observed_stat - 1, n=n_flips, p=0.5)

print(f"Monte Carlo p-value:    {p_value:.3f}")
print(f"Exact binomial p-value: {exact_p:.3f}")

The two values should be close — but the binomial formula is exact and requires no simulation at all.


### The antibiotics example: standard two-group tests

The antibiotics problem is also a well-studied setting: we have two groups and a continuous measurement, and we want to know whether the measurement is independent of the group label. Statisticians have been working on this type of problem for a long time, and several standard tests exist depending on what assumptions you are willing to make about the data.

**If you are willing to assume the values in each group are normally distributed**, the standard choice is the **two-sample t-test** (or Welch's t-test if the groups may have different variances). This has a closed-form analytical solution and is implemented in scipy:


In [ ]:
from scipy.stats import ttest_ind

antibiotic_days = days_to_recovery[received_antibiotics]
no_antibiotic_days = days_to_recovery[~received_antibiotics]

statistic, p_value_ttest = ttest_ind(antibiotic_days, no_antibiotic_days, alternative='less')
print(f"Two-sample t-test p-value: {p_value_ttest:.3f}")

**If you do not want to assume normality** — relying only on the empirical values as they are — the standard non-parametric choice is the **Mann-Whitney U test** (also known as the Wilcoxon rank-sum test). It makes no distributional assumptions and also has an analytical solution:

In [ ]:
from scipy.stats import mannwhitneyu

statistic, p_value_mw = mannwhitneyu(antibiotic_days, no_antibiotic_days, alternative='less')
print(f"Mann-Whitney U test p-value: {p_value_mw:.3f}")

Our permutation test from the exercise is actually closely related to the Mann-Whitney U test — both are non-parametric and rely on the exchangeability of labels under H₀. The permutation test is more flexible (it works with any test statistic), while the Mann-Whitney U test has an analytical solution that is faster to compute.

The general lesson is the same as with probability distributions: **Monte Carlo simulation always works** and requires no mathematical derivation, but for standard problems an analytical solution is usually preferred — it is exact, fast, and already implemented in libraries like scipy. When your problem is non-standard, simulation is often the only practical option.